# 첫번재 시도 

In [2]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from tqdm.auto import tqdm
import logging

# ---------------------------------------------------------
# 1. 설정 및 로깅 (주인님이 확인해주신 정확한 경로 반영)
# ---------------------------------------------------------
class Config:
    # CSV 파일의 정확한 절대 경로 (DataSet 폴더 제외)
    TRAIN_CSV = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\Labels\TrainLabels.csv"
    
    # 비디오 파일들이 들어있는 최상위 폴더 경로
    TRAIN_VIDEO_DIR = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Train"
    
    SAVE_MODEL_PATH = "best_focus_model.pth"
    
    BATCH_SIZE = 4       
    EPOCHS = 15
    SEQ_LENGTH = 10      
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 4      
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ---------------------------------------------------------
# 2. 강력한 중첩 폴더 탐색이 적용된 Dataset
# ---------------------------------------------------------
class DAiSEEDataset(Dataset):
    def __init__(self, csv_file, video_dir, sequence_length=10, transform=None):
        if not os.path.exists(csv_file):
            logger.error(f"CSV 파일을 찾을 수 없습니다: {csv_file}")
            self.data = []
            return
            
        self.df = pd.read_csv(csv_file)
        self.video_dir = video_dir
        self.sequence_length = sequence_length
        self.transform = transform or transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        self.data = self._validate_and_map_videos()

    def _validate_and_map_videos(self):
        valid_data = []
        logger.info(f"[{self.video_dir}] 하위의 모든 비디오 파일을 탐색합니다...")
        
        # 1. 하위 폴더 전체 탐색하여 비디오 경로 매핑
        video_files_map = {}
        for root, dirs, files in os.walk(self.video_dir):
            for file in files:
                if file.lower().endswith(('.mp4', '.avi')):
                    clip_id_only = os.path.splitext(file)[0]
                    video_files_map[clip_id_only] = os.path.join(root, file)

        # 2. CSV 라벨과 매칭
        for idx, row in self.df.iterrows():
            csv_clip_id = str(row['ClipID']).strip()
            if csv_clip_id.lower().endswith(('.mp4', '.avi')):
                csv_clip_id = os.path.splitext(csv_clip_id)[0]
                
            if csv_clip_id in video_files_map:
                engagement_level = int(row['Engagement'])
                valid_data.append({
                    'video_path': video_files_map[csv_clip_id],
                    'label': engagement_level
                })
                
        logger.info(f"총 {len(valid_data)}개의 비디오-라벨 매핑을 완료했습니다.")
        return valid_data

    def extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        frames = []
        
        if not cap.isOpened():
            return torch.zeros((self.sequence_length, 3, 224, 224))

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            cap.release()
            return torch.zeros((self.sequence_length, 3, 224, 224))

        step = max(1, total_frames // self.sequence_length)
        
        for i in range(self.sequence_length):
            cap.set(cv2.CAP_PROP_POS_FRAMES, min(i * step, total_frames - 1))
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = self.transform(frame)
                frames.append(frame)
            else:
                frames.append(torch.zeros((3, 224, 224)))
                
        cap.release()
        return torch.stack(frames)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        frames = self.extract_frames(item['video_path'])
        label = torch.tensor(item['label'], dtype=torch.long)
        return frames, label

# ---------------------------------------------------------
# 3. 모델 정의 (ResNet + LSTM 하이브리드)
# ---------------------------------------------------------
class AttentionFocusModel(nn.Module):
    def __init__(self, num_classes=Config.NUM_CLASSES, hidden_dim=256):
        super(AttentionFocusModel, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        
        self.lstm = nn.LSTM(
            input_size=512, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.3
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        b, seq_len, c, h, w = x.size()
        x = x.view(b * seq_len, c, h, w)
        c_out = self.cnn(x)
        c_out = c_out.view(b, seq_len, -1)
        lstm_out, _ = self.lstm(c_out)
        out = self.fc(lstm_out[:, -1, :])
        return out

# ---------------------------------------------------------
# 4. 학습 파이프라인
# ---------------------------------------------------------
def train_model():
    logger.info(f"학습을 시작합니다. 사용 장치: {Config.DEVICE}")
    
    train_dataset = DAiSEEDataset(Config.TRAIN_CSV, Config.TRAIN_VIDEO_DIR, Config.SEQ_LENGTH)
    
    if len(train_dataset) == 0:
        logger.error("학습할 데이터가 없습니다. 매핑 로직을 확인해 주세요.")
        return

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    
    model = AttentionFocusModel().to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    
    best_loss = float('inf')
    
    for epoch in range(Config.EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}", leave=False)
        
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(Config.DEVICE), labels.to(Config.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            progress_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        
        logger.info(f"Epoch [{epoch+1}/{Config.EPOCHS}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")
        
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), Config.SAVE_MODEL_PATH)
            logger.info(f"[*] 모델 저장 완료 -> {Config.SAVE_MODEL_PATH}")

if __name__ == "__main__":
    train_model()

F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-10 10:18:12,586 [INFO] 학습을 시작합니다. 사용 장치: cuda
2026-03-10 10:18:12,595 [INFO] [F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Train] 하위의 모든 비디오 파일을 탐색합니다...
2026-03-10 10:18:13,831 [INFO] 총 5357개의 비디오-라벨 매핑을 완료했습니다.
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The curre

In [1]:
import torch

print(f"CUDA 활성화 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"인식된 GPU 모델명: {torch.cuda.get_device_name(0)}")
else:
    print("현재 PyTorch가 GPU를 인식하지 못하고 있습니다. (CPU 전용 모드로 동작 중)")

CUDA 활성화 여부: True
인식된 GPU 모델명: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# ---------------------------------------------------------
# 1. 검증용 환경 및 경로 설정 (Validation Data)
# ---------------------------------------------------------
class ValConfig:
    # 이전 캡처에서 확인된 Validation 경로를 적용했습니다.
    VAL_CSV = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\Labels\ValidationLabels.csv"
    VAL_VIDEO_DIR = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation"
    
    LOAD_MODEL_PATH = "best_focus_model.pth"
    BATCH_SIZE = 4
    SEQ_LENGTH = 10
    NUM_CLASSES = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# 2. 검증 평가 파이프라인
# ---------------------------------------------------------
def evaluate_model():
    logger.info("=== 모델 검증(Validation) 평가를 시작합니다 ===")
    
    # 검증 데이터셋 로드 (기존에 정의된 DAiSEEDataset 클래스 재사용)
    val_dataset = DAiSEEDataset(ValConfig.VAL_CSV, ValConfig.VAL_VIDEO_DIR, ValConfig.SEQ_LENGTH)
    
    if len(val_dataset) == 0:
        logger.error("검증할 데이터가 없습니다. Validation 경로를 다시 확인해 주세요.")
        return
        
    val_loader = DataLoader(val_dataset, batch_size=ValConfig.BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 모델 초기화 및 학습된 가중치 불러오기
    model = AttentionFocusModel(num_classes=ValConfig.NUM_CLASSES).to(ValConfig.DEVICE)
    
    try:
        model.load_state_dict(torch.load(ValConfig.LOAD_MODEL_PATH, map_location=ValConfig.DEVICE))
        logger.info(f"성공적으로 가중치를 불러왔습니다: {ValConfig.LOAD_MODEL_PATH}")
    except Exception as e:
        logger.error(f"모델 가중치 로드 실패: {e}")
        return

    # 평가 모드 전환 (Dropout, BatchNorm 등 비활성화)
    model.eval()
    
    all_preds = []
    all_labels = []
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    
    # 진행률 표시줄
    progress_bar = tqdm(val_loader, desc="Validating", leave=False)
    
    # 기울기 계산 비활성화 (메모리 절약 및 속도 향상)
    with torch.no_grad():
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(ValConfig.DEVICE), labels.to(ValConfig.DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            
            # 예측값 추출
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # 최종 결과 계산
    avg_loss = running_loss / len(val_loader)
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    
    overall_accuracy = np.mean(all_preds == all_labels) * 100
    
    print("\n" + "="*50)
    print(f"✅ 검증 완료 - Average Loss: {avg_loss:.4f}")
    print(f"✅ 전체 정확도 (Accuracy): {overall_accuracy:.2f}%")
    print("="*50)
    
    # 클래스별 세부 성능 지표 (Precision, Recall, F1-Score)
    print("\n[클래스별 상세 성능 리포트]")
    target_names = ['Level 0 (지루함/딴짓)', 'Level 1 (약간 집중)', 'Level 2 (보통 집중)', 'Level 3 (매우 집중)']
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))
    
    # 혼동 행렬 (어떤 클래스를 서로 헷갈려하는지 파악)
    print("\n[혼동 행렬 (Confusion Matrix)]")
    print(confusion_matrix(all_labels, all_preds))

if __name__ == "__main__":
    evaluate_model()

2026-03-10 14:26:05,938 [INFO] === 모델 검증(Validation) 평가를 시작합니다 ===
2026-03-10 14:26:05,944 [INFO] [F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation] 하위의 모든 비디오 파일을 탐색합니다...
2026-03-10 14:26:06,261 [INFO] 총 1429개의 비디오-라벨 매핑을 완료했습니다.
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\msi\AppData\Local\Temp\ipykernel_21752\1166298784.py:41: 


✅ 검증 완료 - Average Loss: 1.3644
✅ 전체 정확도 (Accuracy): 49.20%

[클래스별 상세 성능 리포트]
                  precision    recall  f1-score   support

Level 0 (지루함/딴짓)       0.00      0.00      0.00        23
 Level 1 (약간 집중)       0.00      0.00      0.00       143
 Level 2 (보통 집중)       0.58      0.58      0.58       813
 Level 3 (매우 집중)       0.38      0.52      0.44       450

        accuracy                           0.49      1429
       macro avg       0.24      0.27      0.25      1429
    weighted avg       0.45      0.49      0.47      1429


[혼동 행렬 (Confusion Matrix)]
[[  0   0  22   1]
 [  0   0 106  37]
 [  0   0 470 343]
 [  0   0 217 233]]


# 두번째 시도 

In [ ]:
import numpy as np

# ---------------------------------------------------------
# 1. 설정 및 로깅 (에포크 100 및 검증 경로 추가)
# ---------------------------------------------------------
class Config:
    TRAIN_CSV = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\Labels\TrainLabels.csv"
    TRAIN_VIDEO_DIR = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Train"
    
    # [추가됨] 조기 종료를 위해 훈련 중에도 검증 데이터가 필요합니다.
    VAL_CSV = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\Labels\ValidationLabels.csv"
    VAL_VIDEO_DIR = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\DAiSEE\DataSet\Validation"
    
    SAVE_MODEL_PATH = "best_focus_model.pth"
    
    BATCH_SIZE = 4       
    EPOCHS = 100         # 주인님의 명령에 따라 100으로 상향
    PATIENCE = 15        # 15번 연속으로 최고 기록을 깨지 못하면 조기 종료
    SEQ_LENGTH = 10      
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 4      
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ... (DAiSEEDataset 클래스와 AttentionFocusModel 클래스는 기존과 동일하게 유지합니다) ...

# ---------------------------------------------------------
# 4. 불균형 해소 및 조기 종료가 적용된 학습 파이프라인
# ---------------------------------------------------------
def train_model():
    logger.info(f"학습을 시작합니다. 사용 장치: {Config.DEVICE}")
    
    # 1. 훈련 및 검증 데이터 로드
    train_dataset = DAiSEEDataset(Config.TRAIN_CSV, Config.TRAIN_VIDEO_DIR, Config.SEQ_LENGTH)
    val_dataset = DAiSEEDataset(Config.VAL_CSV, Config.VAL_VIDEO_DIR, Config.SEQ_LENGTH)
    
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        logger.error("데이터가 없습니다. 경로를 확인해 주세요.")
        return

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 2. 동적 클래스 가중치 계산 (Class Imbalance 해결의 핵심)
    logger.info("데이터 클래스 분포를 분석하여 가중치를 계산합니다...")
    label_counts = [0] * Config.NUM_CLASSES
    for item in train_dataset.data:
        label_counts[item['label']] += 1
        
    total_samples = sum(label_counts)
    # 데이터가 적은 클래스일수록 가중치가 기하급수적으로 커짐
    class_weights = [total_samples / (Config.NUM_CLASSES * max(count, 1)) for count in label_counts]
    class_weights_tensor = torch.FloatTensor(class_weights).to(Config.DEVICE)
    
    logger.info(f"산출된 패널티 가중치: Level 0({class_weights[0]:.2f}), Level 1({class_weights[1]:.2f}), Level 2({class_weights[2]:.2f}), Level 3({class_weights[3]:.2f})")
    
    model = AttentionFocusModel().to(Config.DEVICE)
    
    # 3. 가중치가 부여된 손실 함수 적용
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    # 4. 100 Epoch 학습 및 조기 종료(Early Stopping) 루프
    for epoch in range(Config.EPOCHS):
        # --- 훈련(Training) 페이즈 ---
        model.train()
        running_train_loss = 0.0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(Config.DEVICE), labels.to(Config.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
            train_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        # --- 검증(Validation) 페이즈 ---
        model.eval()
        running_val_loss = 0.0
        
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS} [Val]", leave=False)
        with torch.no_grad():
            for val_inputs, val_labels in val_bar:
                val_inputs, val_labels = val_inputs.to(Config.DEVICE), val_labels.to(Config.DEVICE)
                val_outputs = model(val_inputs)
                v_loss = criterion(val_outputs, val_labels)
                running_val_loss += v_loss.item()
                
        avg_val_loss = running_val_loss / len(val_loader)
        
        logger.info(f"Epoch [{epoch+1}/{Config.EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        # --- 최적 모델 판별 및 조기 종료 로직 ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), Config.SAVE_MODEL_PATH)
            logger.info(f"✨ 최고 성능 갱신! 모델 저장 완료 -> {Config.SAVE_MODEL_PATH} (Val Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            logger.info(f"성능 개선 없음 ({patience_counter}/{Config.PATIENCE})")
            
        if patience_counter >= Config.PATIENCE:
            logger.info(f"🚨 {Config.PATIENCE}회 연속 성능 개선이 없어 과적합을 방지하기 위해 학습을 조기 종료합니다.")
            break
            
    logger.info("모든 학습 프로세스가 성공적으로 완료되었습니다.")

if __name__ == "__main__":
    train_model()

# FER

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
import logging

# ---------------------------------------------------------
# 1. 환경 설정 및 경로 (FER2013 전용)
# ---------------------------------------------------------
class FERConfig:
    # 주인님께서 지정하신 FER2013 최상위 경로
    BASE_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\fer2013"
    
    # 일반적인 FER2013 폴더 구조 (train / test 하위에 감정별 폴더가 있다고 가정)
    TRAIN_DIR = os.path.join(BASE_PATH, "train")
    VAL_DIR = os.path.join(BASE_PATH, "test") 
    
    SAVE_MODEL_PATH = "best_fer_model.pth"
    
    # 흑백 단일 프레임이므로 배치 사이즈를 넉넉하게 잡을 수 있습니다.
    BATCH_SIZE = 64       
    EPOCHS = 50
    PATIENCE = 10         # 조기 종료 조건
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 7       # 7가지 감정
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ---------------------------------------------------------
# 2. 모델 정의 (ResNet18 전이 학습)
# ---------------------------------------------------------
class FERModel(nn.Module):
    def __init__(self, num_classes=FERConfig.NUM_CLASSES):
        super(FERModel, self).__init__()
        # 사전 학습된 ResNet18 로드
        self.resnet = models.resnet18(pretrained=True)
        
        # 마지막 출력층을 7개(감정 클래스)로 교체
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(0.5), # 과적합 방지를 위한 드롭아웃
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.resnet(x)

# ---------------------------------------------------------
# 3. 학습 파이프라인 엔진
# ---------------------------------------------------------
def train_fer_model():
    logger.info(f"FER 모델 학습을 시작합니다. 사용 장치: {FERConfig.DEVICE}")
    
    # FER 데이터는 정지 이미지이므로 강력한 데이터 증강(Augmentation)이 필요합니다.
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.Resize((224, 224)),      # ResNet 입력 크기에 맞춤
        transforms.Grayscale(num_output_channels=3), # 흑백 이미지를 3채널 RGB로 복제
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # ImageFolder를 사용하여 폴더명(감정)을 자동으로 라벨링
    try:
        train_dataset = datasets.ImageFolder(root=FERConfig.TRAIN_DIR, transform=train_transform)
        val_dataset = datasets.ImageFolder(root=FERConfig.VAL_DIR, transform=val_transform)
    except FileNotFoundError:
        logger.error(f"데이터 폴더를 찾을 수 없습니다. 경로 구조를 확인하십시오: {FERConfig.TRAIN_DIR}")
        return

    logger.info(f"학습 이미지: {len(train_dataset)}장, 검증 이미지: {len(val_dataset)}장 로드 완료.")
    logger.info(f"감정 클래스 매핑: {train_dataset.class_to_idx}")

    train_loader = DataLoader(train_dataset, batch_size=FERConfig.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=FERConfig.BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = FERModel().to(FERConfig.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=FERConfig.LEARNING_RATE, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(FERConfig.EPOCHS):
        # --- 훈련(Train) ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{FERConfig.EPOCHS} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            train_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            
        train_acc = 100 * correct / total
        
        # --- 검증(Validation) ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        
        logger.info(f"Epoch [{epoch+1}/{FERConfig.EPOCHS}] - Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}% | Val Loss: {avg_val_loss:.4f}")
        
        # --- 조기 종료 및 최적 모델 저장 ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), FERConfig.SAVE_MODEL_PATH)
            logger.info(f"✨ 최고 성능 갱신! 모델 저장됨 -> {FERConfig.SAVE_MODEL_PATH}")
        else:
            patience_counter += 1
            
        if patience_counter >= FERConfig.PATIENCE:
            logger.info(f"🚨 {FERConfig.PATIENCE}회 연속 개선 없음. 학습 조기 종료.")
            break

if __name__ == "__main__":
    train_fer_model()

2026-03-10 14:41:04,827 [INFO] FER 모델 학습을 시작합니다. 사용 장치: cuda
2026-03-10 14:41:04,956 [INFO] 학습 이미지: 28709장, 검증 이미지: 7178장 로드 완료.
2026-03-10 14:41:04,957 [INFO] 감정 클래스 매핑: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
2026-03-10 14:43:42,123 [INFO] Epoch [1/50] - Train Acc: 49.25%

# FER test2

In [9]:
import torchvision.transforms as transforms
import torch.optim as optim
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets

# ---------------------------------------------------------
# 1. 환경 설정 (조기 종료 3회로 엄격하게 수정)
# ---------------------------------------------------------
class FERConfig:
    BASE_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\fer2013"
    TRAIN_DIR = os.path.join(BASE_PATH, "train")
    VAL_DIR = os.path.join(BASE_PATH, "test") 
    
    SAVE_MODEL_PATH = "best_fer_model.pth"
    BATCH_SIZE = 64       
    EPOCHS = 50
    PATIENCE = 3          # 주인님의 명령에 따라 3회 연속 개선 없으면 즉시 종료
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 7       
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# 2. 실무급 FER 훈련 파이프라인
# ---------------------------------------------------------
def train_production_fer_model():
    logger.info("실무 프로덕션급 FER 모델 훈련 파이프라인을 가동합니다. (조기 종료: 3회)")
    
    # 데이터 증강 (Random Erasing 포함)
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0) 
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = datasets.ImageFolder(root=FERConfig.TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(root=FERConfig.VAL_DIR, transform=val_transform)
    train_loader = DataLoader(train_dataset, batch_size=FERConfig.BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=FERConfig.BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = FERModel().to(FERConfig.DEVICE)
    
    # 라벨 스무딩 적용
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) 
    optimizer = optim.AdamW(model.parameters(), lr=FERConfig.LEARNING_RATE, weight_decay=1e-3)
    
    # 스케줄러: 조기 종료가 3회이므로, 스케줄러는 2회 만에 더 빨리 반응하도록 세팅
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)
    
    best_val_loss = float('inf')
    early_stop_counter = 0
    
    for epoch in range(FERConfig.EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{FERConfig.EPOCHS} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            train_bar.set_postfix({'Loss': f'{loss.item():.4f}'})
            
        train_acc = 100 * correct / total
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        
        logger.info(f"Epoch [{epoch+1}/{FERConfig.EPOCHS}] - Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {avg_val_loss:.4f}")
        
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            torch.save(model.state_dict(), FERConfig.SAVE_MODEL_PATH)
            logger.info(f"✨ 최고 성능 갱신! 모델 저장됨 -> {FERConfig.SAVE_MODEL_PATH}")
        else:
            early_stop_counter += 1
            logger.info(f"⚠️ 성능 개선 없음 ({early_stop_counter}/{FERConfig.PATIENCE})")
            
        if early_stop_counter >= FERConfig.PATIENCE:
            logger.info(f"🚨 {FERConfig.PATIENCE}회 연속 개선이 없어 과적합 방지를 위해 학습을 즉시 마감합니다.")
            break

if __name__ == "__main__":
    train_production_fer_model()

2026-03-10 15:42:15,019 [INFO] 실무 프로덕션급 FER 모델 훈련 파이프라인을 가동합니다. (조기 종료: 3회)
2026-03-10 15:45:32,258 [INFO] Epoch [1/50] - Train Acc: 46.12% | Val Acc: 59.72% | Val Loss: 1.2619                                          
2026-03-10 15:45:32,337 [INFO] ✨ 최고 성능 갱신! 모델 저장됨 -> best_fer_model.pth
2026-03-10 15:48:38,505 [INFO] Epoch [2/50] - Train Acc: 57.76% | Val Acc: 63.21% | Val Loss: 1.1974                                          
2026-03-10 15:48:38,579 [INFO] ✨ 최고 성능 갱신! 모델 저장됨 -> best_fer_model.pth
2026-03-10 15:51:55,223 [INFO] Epoch [3/50] - Train Acc: 61.83% | Val Acc: 65.12% | Val Loss: 1.1599                                          
2026-03-10 15:51:55,321 [INFO] ✨ 최고 성능 갱신! 모델 저장됨 -> best_fer_model.pth
2026-03-10 15:55:02,971 [INFO] Epoch [4/50] - Train Acc: 64.29% | Val Acc: 65.88% | Val Loss: 1.1430                                          
2026-03-10 15:55:03,069 [INFO] ✨ 최고 성능 갱신! 모델 저장됨 -> best_fer_model.pth
2026-03-10 15:58:11,357 [INFO] Epoch [5/50] - Train Acc: 66.38% 

# fer test3

In [11]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
import logging

# ---------------------------------------------------------
# 1. 환경 설정 (64GB RAM & ResNet50 최적화)
# ---------------------------------------------------------
class FERConfig:
    BASE_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\fer2013"
    TRAIN_DIR = os.path.join(BASE_PATH, "train")
    VAL_DIR = os.path.join(BASE_PATH, "test") 
    SAVE_MODEL_PATH = "best_fer_resnet50_centerloss.pth"
    
    BATCH_SIZE = 64       
    EPOCHS = 50
    PATIENCE = 3          # 3회 연속 개선 없으면 종료
    LEARNING_RATE = 1e-4
    CENTER_LR = 0.5       # Center Loss 전용 학습률
    CENTER_LOSS_WEIGHT = 0.007 # Softmax와 Center Loss 사이의 균형 가중치
    FEAT_DIM = 512        # 특징 추출 차원
    NUM_CLASSES = 7       
    NUM_WORKERS = 8       # 64GB RAM 활용
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ---------------------------------------------------------
# 2. Center Loss 정의 (클래스 내 응집력 및 클래스 간 분리 강화)
# ---------------------------------------------------------
class CenterLoss(nn.Module):
    def __init__(self, num_classes=7, feat_dim=512, use_gpu=True):
        super(CenterLoss, self).__init__()
        self.num_classes = num_classes
        self.feat_dim = feat_dim
        self.use_gpu = use_gpu
        if self.use_gpu:
            self.centers = nn.Parameter(torch.randn(num_classes, feat_dim).cuda())
        else:
            self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, x, labels):
        batch_size = x.size(0)
        distmat = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(batch_size, self.num_classes) + \
                  torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(self.num_classes, batch_size).t()
        distmat.addmm_(1, -2, x, self.centers.t())

        classes = torch.arange(self.num_classes).long()
        if self.use_gpu: classes = classes.cuda()
        labels = labels.unsqueeze(1).expand(batch_size, self.num_classes)
        mask = labels.eq(classes.expand(batch_size, self.num_classes))

        dist = distmat * mask.float()
        loss = dist.clamp(min=1e-12, max=1e+12).sum() / batch_size
        return loss

# ---------------------------------------------------------
# 3. 모델 정의 (Feature 추출 레이어 분리형 ResNet50)
# ---------------------------------------------------------
class FERModel(nn.Module):
    def __init__(self, num_classes=FERConfig.NUM_CLASSES):
        super(FERModel, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity() # 기존 FC 제거
        
        # Center Loss에 태울 특징 벡터(512차원) 추출층
        self.feature_layer = nn.Sequential(
            nn.Linear(in_features, FERConfig.FEAT_DIM),
            nn.BatchNorm1d(FERConfig.FEAT_DIM),
            nn.ReLU()
        )
        self.classifier = nn.Linear(FERConfig.FEAT_DIM, num_classes)

    def forward(self, x):
        features = self.resnet(x)
        features = self.feature_layer(features)
        logits = self.classifier(features)
        return features, logits

# ---------------------------------------------------------
# 4. 통합 훈련 및 검증 엔진
# ---------------------------------------------------------
def train_center_loss_fer():
    logger.info(f"[*] ResNet50 + Center Loss 프로덕션 훈련 시작. (Device: {FERConfig.DEVICE})")
    
    # 데이터 증강 설정
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.5, value='random') 
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 64GB RAM 최적화 데이터 로더
    train_loader = DataLoader(datasets.ImageFolder(FERConfig.TRAIN_DIR, train_transform), 
                              batch_size=FERConfig.BATCH_SIZE, shuffle=True, 
                              num_workers=FERConfig.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(datasets.ImageFolder(FERConfig.VAL_DIR, val_transform), 
                            batch_size=FERConfig.BATCH_SIZE, shuffle=False, 
                            num_workers=FERConfig.NUM_WORKERS, pin_memory=True)

    model = FERModel().to(FERConfig.DEVICE)
    criterion_ce = nn.CrossEntropyLoss(label_smoothing=0.1)
    criterion_center = CenterLoss(num_classes=FERConfig.NUM_CLASSES, feat_dim=FERConfig.FEAT_DIM).to(FERConfig.DEVICE)
    
    # 모델용 AdamW와 Center용 Adam 옵티마이저 분리
    optimizer_model = optim.AdamW(model.parameters(), lr=FERConfig.LEARNING_RATE, weight_decay=1e-3)
    optimizer_center = optim.Adam(criterion_center.parameters(), lr=FERConfig.CENTER_LR)
    
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_model, mode='min', factor=0.5, patience=2, verbose=True)

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(FERConfig.EPOCHS):
        # --- 훈련 단계 ---
        model.train()
        train_loss, correct, total = 0, 0, 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{FERConfig.EPOCHS} [Train]", leave=False)
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
            
            optimizer_model.zero_grad()
            optimizer_center.zero_grad()
            
            features, outputs = model(inputs)
            loss_ce = criterion_ce(outputs, labels)
            loss_center = criterion_center(features, labels)
            
            # 하이브리드 손실 계산
            loss = loss_ce + (FERConfig.CENTER_LOSS_WEIGHT * loss_center)
            
            loss.backward()
            optimizer_model.step()
            
            # Center Loss 파라미터 업데이트를 위한 수동 기울기 스케일링
            for param in criterion_center.parameters():
                param.grad.data *= (1. / FERConfig.CENTER_LOSS_WEIGHT)
            optimizer_center.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            train_bar.set_postfix({'CE': f'{loss_ce.item():.4f}', 'Center': f'{loss_center.item():.4f}'})
            
        train_acc = 100 * correct / total
        
        # --- 검증 단계 ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
                features, outputs = model(inputs)
                
                v_loss_ce = criterion_ce(outputs, labels)
                v_loss_center = criterion_center(features, labels)
                v_loss = v_loss_ce + (FERConfig.CENTER_LOSS_WEIGHT * v_loss_center)
                
                val_loss += v_loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        
        logger.info(f"Epoch [{epoch+1}/{FERConfig.EPOCHS}] - Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {avg_val_loss:.4f}")
        
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            # 모델 저장 시 classifier와 feature_layer를 포함한 전체 가중치 저장
            torch.save(model.state_dict(), FERConfig.SAVE_MODEL_PATH)
            logger.info(f"✨ 최고 성능 갱신! 모델 저장됨 -> {FERConfig.SAVE_MODEL_PATH}")
        else:
            patience_counter += 1
            logger.info(f"⚠️ 성능 개선 없음 ({patience_counter}/{FERConfig.PATIENCE})")
            
        if patience_counter >= FERConfig.PATIENCE:
            logger.info(f"🚨 {FERConfig.PATIENCE}회 연속 개선이 없어 학습을 종료합니다.")
            break

if __name__ == "__main__":
    train_center_loss_fer()

2026-03-10 17:04:32,437 [INFO] [*] ResNet50 + Center Loss 프로덕션 훈련 시작. (Device: cuda)
Epoch 1/50 [Train]:   0%|                                                                                             | 0/449 [00:00<?, ?it/s]C:\Users\msi\AppData\Local\Temp\ipykernel_21752\2846608240.py:51: UserWarning: This overload of addmm_ is deprecated:
	addmm_(Number beta, Number alpha, Tensor mat1, Tensor mat2)
Consider using one of the following signatures instead:
	addmm_(Tensor mat1, Tensor mat2, *, Number beta = 1, Number alpha = 1) (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\python_arg_parser.cpp:1642.)
  distmat.addmm_(1, -2, x, self.centers.t())
2026-03-10 17:08:43,987 [INFO] Epoch [1/50] - Train Acc: 48.91% | Val Acc: 60.37% | Val Loss: 1.3162                                          
2026-03-10 17:08:44,113 [INFO] ✨ 최고 성능 갱신! 모델 저장됨 -> best_fer_resnet50_centerloss.pth
2026-03-10 17:12:57,733 [INFO] Epoch [2/50] - Train Acc: 

# 밤동안 돌리는 test 4

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm
import logging

# ---------------------------------------------------------
# 1. 환경 설정 (0.5 미만 타겟팅 최적화)
# ---------------------------------------------------------
class FERConfig:
    BASE_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\fer2013"
    TRAIN_DIR = os.path.join(BASE_PATH, "train")
    VAL_DIR = os.path.join(BASE_PATH, "test") 
    SAVE_MODEL_PATH = "best_fer_resnet50_ohem.pth"
    
    BATCH_SIZE = 64       
    EPOCHS = 50
    PATIENCE = 3          # 주인님 지시: 3회 연속 개선 없으면 종료
    LEARNING_RATE = 1e-4
    CENTER_LR = 0.5       
    CENTER_LOSS_WEIGHT = 0.005 # Center Loss 비중 미세 조정
    FEAT_DIM = 512        
    NUM_CLASSES = 7       
    NUM_WORKERS = 8       # 64GB RAM 활용
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ---------------------------------------------------------
# 2. OHEM (Online Hard Example Mining) Loss 정의
# ---------------------------------------------------------
class OHEMLoss(nn.Module):
    def __init__(self, ratio=0.7): # 상위 70%의 어려운 데이터에만 집중
        super(OHEMLoss, self).__init__()
        self.ratio = ratio

    def forward(self, logits, targets):
        # 모든 샘플의 CE Loss 계산 (Label Smoothing 0.05로 정답 확신도 상향)
        ce_loss = F.cross_entropy(logits, targets, reduction='none', label_smoothing=0.05)
        
        # Loss가 높은(어려운) 샘플의 개수 산출
        num_inst = ce_loss.numel()
        num_hards = int(num_inst * self.ratio)
        
        # 상위 num_hards개의 Loss만 평균내어 반환
        val, _ = torch.topk(ce_loss, num_hards)
        return val.mean()

# ---------------------------------------------------------
# 3. Center Loss & FERModel (ResNet50)
# ---------------------------------------------------------
class CenterLoss(nn.Module):
    def __init__(self, num_classes=7, feat_dim=512):
        super(CenterLoss, self).__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim).cuda())

    def forward(self, x, labels):
        batch_size = x.size(0)
        distmat = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(batch_size, 7) + \
                  torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(7, batch_size).t()
        distmat.addmm_(1, -2, x, self.centers.t())
        classes = torch.arange(7).long().cuda()
        labels_exp = labels.unsqueeze(1).expand(batch_size, 7)
        mask = labels_exp.eq(classes.expand(batch_size, 7))
        dist = distmat * mask.float()
        return dist.clamp(min=1e-12, max=1e+12).sum() / batch_size

class FERModel(nn.Module):
    def __init__(self, num_classes=7):
        super(FERModel, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()
        self.feature_layer = nn.Sequential(
            nn.Linear(in_features, FERConfig.FEAT_DIM),
            nn.BatchNorm1d(FERConfig.FEAT_DIM),
            nn.ReLU()
        )
        self.classifier = nn.Linear(FERConfig.FEAT_DIM, num_classes)

    def forward(self, x):
        f = self.resnet(x)
        f = self.feature_layer(f)
        l = self.classifier(f)
        return f, l

# ---------------------------------------------------------
# 4. 통합 훈련 엔진 (OHEM 적용)
# ---------------------------------------------------------
def train_ohem_fer():
    # 데이터 증강 (64GB RAM 기반 고속 로딩)
    train_loader = DataLoader(datasets.ImageFolder(FERConfig.TRAIN_DIR, transforms.Compose([
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(15), transforms.Resize((224, 224)),
        transforms.Grayscale(3), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.5, value='random')])), 
        batch_size=FERConfig.BATCH_SIZE, shuffle=True, num_workers=FERConfig.NUM_WORKERS, pin_memory=True)
    
    val_loader = DataLoader(datasets.ImageFolder(FERConfig.VAL_DIR, transforms.Compose([
        transforms.Resize((224, 224)), transforms.Grayscale(3), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])), 
        batch_size=FERConfig.BATCH_SIZE, shuffle=False, num_workers=FERConfig.NUM_WORKERS, pin_memory=True)

    model = FERModel().to(FERConfig.DEVICE)
    criterion_ohem = OHEMLoss(ratio=0.7) # 하위 30% 쉬운 문제는 과감히 버림
    criterion_center = CenterLoss().to(FERConfig.DEVICE)
    
    optimizer_model = optim.AdamW(model.parameters(), lr=FERConfig.LEARNING_RATE, weight_decay=1e-3)
    optimizer_center = optim.Adam(criterion_center.parameters(), lr=FERConfig.CENTER_LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer_model, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    patience_cnt = 0

    for epoch in range(FERConfig.EPOCHS):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [OHEM Training]"):
            inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
            optimizer_model.zero_grad(); optimizer_center.zero_grad()
            
            features, outputs = model(inputs)
            loss_ohem = criterion_ohem(outputs, labels) # 어려운 데이터 집중 공격
            loss_center = criterion_center(features, labels)
            loss = loss_ohem + (FERConfig.CENTER_LOSS_WEIGHT * loss_center)
            
            loss.backward()
            optimizer_model.step()
            for p in criterion_center.parameters(): p.grad.data *= (1./FERConfig.CENTER_LOSS_WEIGHT)
            optimizer_center.step()
            
            train_loss += loss.item()
            _, pred = torch.max(outputs, 1); total += labels.size(0); correct += (pred == labels).sum().item()

        # 검증 루프 (0.5 타겟 확인용)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(FERConfig.DEVICE), labels.to(FERConfig.DEVICE)
                features, outputs = model(inputs)
                v_loss = F.cross_entropy(outputs, labels) # 검증은 표준 CE로 측정
                val_loss += v_loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        logger.info(f"Epoch [{epoch+1}] Val Loss: {avg_val_loss:.4f} (Target: 0.5)")
        
        scheduler.step(avg_val_loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss; patience_cnt = 0
            torch.save(model.state_dict(), FERConfig.SAVE_MODEL_PATH)
            logger.info("✨ 최고 기록 갱신!")
        else:
            patience_cnt += 1
        if patience_cnt >= FERConfig.PATIENCE: break

if __name__ == "__main__":
    train_ohem_fer()

F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1 [OHEM Training]:   0%|                     

# 3/11 test 시작

In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm # ipywidgets 경고 방지를 위해 표준 tqdm 사용
import logging

# 1. 64GB RAM & 고성능 하드웨어 설정
class FERConfig:
    BASE_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\fer2013"
    TRAIN_DIR = os.path.join(BASE_PATH, "train")
    VAL_DIR = os.path.join(BASE_PATH, "test") 
    SAVE_PATH = "best_fer_ultra_final.pth"
    
    BATCH_SIZE = 64       
    EPOCHS = 50
    PATIENCE = 3          # 주인님 명령: 3회 정체 시 즉시 종료
    LR = 1e-4
    CENTER_LR = 0.5       
    CENTER_WEIGHT = 0.007 # Center Loss 반영 비율
    FEAT_DIM = 512        
    NUM_CLASSES = 7       
    NUM_WORKERS = 8       # 64GB RAM 기반 고속 로딩
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # [Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral]
    # 옆 사람의 0.3을 잡기 위해 Angry(0)와 Sad(4)에 압도적 벌금을 매깁니다.
    CLASS_WEIGHTS = torch.tensor([3.5, 4.0, 2.5, 1.0, 2.5, 1.5, 0.6]).to(DEVICE)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# 2. 클래스별 가중치가 적용된 OHEM Loss (어려운 샘플 집중 타격)
class WeightedOHEMLoss(nn.Module):
    def __init__(self, ratio=0.7):
        super().__init__()
        self.ratio = ratio

    def forward(self, logits, targets):
        # 가중치 + 라벨 스무딩(0.05)으로 정답 확신도와 오답 패널티를 동시에 잡습니다.
        ce_loss = F.cross_entropy(logits, targets, reduction='none', 
                                  weight=FERConfig.CLASS_WEIGHTS, label_smoothing=0.05)
        num_hards = int(ce_loss.numel() * self.ratio)
        val, _ = torch.topk(ce_loss, num_hards)
        return val.mean()

# 3. Center Loss (클래스 간 응집력 및 분리 강화)
class CenterLoss(nn.Module):
    def __init__(self, num_classes=7, feat_dim=512):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim).to(FERConfig.DEVICE))

    def forward(self, x, labels):
        batch_size = x.size(0)
        distmat = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(batch_size, 7) + \
                  torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(7, batch_size).t()
        distmat.addmm_(1, -2, x, self.centers.t())
        classes = torch.arange(7).long().to(FERConfig.DEVICE)
        labels_exp = labels.unsqueeze(1).expand(batch_size, 7)
        mask = labels_exp.eq(classes.expand(batch_size, 7))
        return (distmat * mask.float()).clamp(min=1e-12, max=1e+12).sum() / batch_size

# 4. ResNet50 기반 모델 (Feature 추출층 분리)
class FERModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet50(weights='IMAGENET1K_V1')
        in_feats = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()
        self.feature_layer = nn.Sequential(
            nn.Linear(in_feats, FERConfig.FEAT_DIM),
            nn.BatchNorm1d(FERConfig.FEAT_DIM),
            nn.ReLU()
        )
        self.classifier = nn.Linear(FERConfig.FEAT_DIM, 7)

    def forward(self, x):
        f = self.resnet(x)
        f = self.feature_layer(f)
        return f, self.classifier(f)

# 5. 통합 훈련 엔진
def train_fer_final():
    logger.info(f"[*] FER ResNet50 Ultra Final 가동. (Device: {FERConfig.DEVICE})")
    
    # 고강도 데이터 증강
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(20), transforms.Resize((224,224)),
        transforms.Grayscale(3), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.5, value='random')
    ])
    val_transform = transforms.Compose([
        transforms.Resize((224,224)), transforms.Grayscale(3), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    train_loader = DataLoader(datasets.ImageFolder(FERConfig.TRAIN_DIR, train_transform), 
                              batch_size=FERConfig.BATCH_SIZE, shuffle=True, 
                              num_workers=FERConfig.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(datasets.ImageFolder(FERConfig.VAL_DIR, val_transform), 
                            batch_size=FERConfig.BATCH_SIZE, shuffle=False, 
                            num_workers=FERConfig.NUM_WORKERS, pin_memory=True)

    model = FERModel().to(FERConfig.DEVICE)
    criterion_ohem = WeightedOHEMLoss().to(FERConfig.DEVICE)
    criterion_center = CenterLoss().to(FERConfig.DEVICE)
    
    optimizer_model = optim.AdamW(model.parameters(), lr=FERConfig.LR, weight_decay=1e-3)
    optimizer_center = optim.Adam(criterion_center.parameters(), lr=FERConfig.CENTER_LR)
    # 웅덩이 탈출용 코사인 스케줄러
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer_model, T_0=10, T_mult=2)

    best_val_loss = float('inf')
    early_stop_cnt = 0

    for epoch in range(FERConfig.EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for imgs, lbls in pbar:
            imgs, lbls = imgs.to(FERConfig.DEVICE), lbls.to(FERConfig.DEVICE)
            optimizer_model.zero_grad(); optimizer_center.zero_grad()
            
            feats, outs = model(imgs)
            l_ohem = criterion_ohem(outs, lbls)
            l_center = criterion_center(feats, lbls)
            loss = l_ohem + (FERConfig.CENTER_WEIGHT * l_center)
            
            loss.backward()
            optimizer_model.step()
            for p in criterion_center.parameters(): p.grad.data *= (1./FERConfig.CENTER_WEIGHT)
            optimizer_center.step()
            
            train_loss += loss.item()
            _, pred = torch.max(outs, 1); train_total += lbls.size(0); train_correct += (pred == lbls).sum().item()
            pbar.set_postfix({'L': f'{loss.item():.4f}', 'Acc': f'{100*train_correct/train_total:.2f}%'})

        # 검증 루프
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(FERConfig.DEVICE), lbls.to(FERConfig.DEVICE)
                _, outs = model(imgs)
                v_loss = F.cross_entropy(outs, lbls) # 검증은 표준 CE
                val_loss += v_loss.item()
                _, pred = torch.max(outs, 1); val_total += lbls.size(0); val_correct += (pred == lbls).sum().item()

        avg_v_loss = val_loss / len(val_loader)
        v_acc = 100 * val_correct / val_total
        logger.info(f"[*] Val Acc: {v_acc:.2f}% | Val Loss: {avg_v_loss:.4f}")
        
        scheduler.step()
        if avg_v_loss < best_val_loss:
            best_val_loss = avg_v_loss; early_stop_cnt = 0
            torch.save(model.state_dict(), FERConfig.SAVE_PATH)
            logger.info("✨ 최고 기록 갱신 및 저장 완료!")
        else:
            early_stop_cnt += 1
        if early_stop_cnt >= FERConfig.PATIENCE: break

if __name__ == "__main__":
    train_fer_final()

2026-03-11 09:08:57,538 [INFO] [*] FER ResNet50 Ultra Final 가동. (Device: cuda)
Epoch 1:   0%|                                                                                                        | 0/449 [00:00<?, ?it/s]C:\Users\msi\AppData\Local\Temp\ipykernel_10156\1092823592.py:60: UserWarning: This overload of addmm_ is deprecated:
	addmm_(Number beta, Number alpha, Tensor mat1, Tensor mat2)
Consider using one of the following signatures instead:
	addmm_(Tensor mat1, Tensor mat2, *, Number beta = 1, Number alpha = 1) (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\python_arg_parser.cpp:1642.)
  distmat.addmm_(1, -2, x, self.centers.t())
Epoch 1: 100%|████████████████████████████████████████████████████████████████████████| 449/449 [03:07<00:00,  2.40it/s, L=3.9646, Acc=25.44%]
2026-03-11 09:12:38,091 [INFO] [*] Val Acc: 31.46% | Val Loss: 1.7110
2026-03-11 09:12:38,235 [INFO] ✨ 최고 기록 갱신 및 저장 완료!
Epoch 2: 100%|██████████████

In [ ]:
import os, torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim, logging
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

# 1. 고성능 환경 및 Loss 타격 설정
class FERConfig:
    BASE_PATH = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\fer2013"
    TRAIN_DIR = os.path.join(BASE_PATH, "train")
    VAL_DIR = os.path.join(BASE_PATH, "test") 
    SAVE_PATH = "best_fer_ultra_final.pth"
    
    BATCH_SIZE = 64       
    EPOCHS = 50
    PATIENCE = 5          # 1.1의 늪을 탈출할 시간을 주기 위해 5로 상향
    LR = 5e-5             # 1.1 벽을 뚫기 위한 정교한 학습률 (상향 조정 금지)
    CENTER_LR = 0.5       
    CENTER_WEIGHT = 0.007 
    FEAT_DIM = 512        
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # [Angry, Disgust, Fear, Happy, Sad, Surprise, Neutral]
    # 주인님, Angry(0)와 Fear(2)에 압도적 벌금을 매겨 1.1의 벽을 부숩니다.
    CLASS_WEIGHTS = torch.tensor([5.0, 4.5, 3.5, 0.5, 3.0, 1.5, 0.5]).to(DEVICE)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# 2. 징벌적 OHEM Loss (레이블 스무딩 제거 버전)
class WeightedOHEMLoss(nn.Module):
    def __init__(self, ratio=0.7):
        super().__init__()
        self.ratio = ratio
    def forward(self, logits, targets):
        # 0.5 미만을 위해 label_smoothing을 0으로 제거하여 정답에 강하게 집착하게 합니다.
        ce_loss = F.cross_entropy(logits, targets, reduction='none', 
                                  weight=FERConfig.CLASS_WEIGHTS, label_smoothing=0.0)
        num_hards = int(ce_loss.numel() * self.ratio)
        val, _ = torch.topk(ce_loss, num_hards)
        return val.mean()

# 3. Center Loss (클래스 간 거리 강제 확장)
class CenterLoss(nn.Module):
    def __init__(self, num_classes=7, feat_dim=512):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim).to(FERConfig.DEVICE))
    def forward(self, x, labels):
        batch_size = x.size(0)
        distmat = torch.pow(x, 2).sum(dim=1, keepdim=True).expand(batch_size, 7) + \
                  torch.pow(self.centers, 2).sum(dim=1, keepdim=True).expand(7, batch_size).t()
        distmat.addmm_(1, -2, x, self.centers.t())
        classes = torch.arange(7).long().to(FERConfig.DEVICE)
        labels_exp = labels.unsqueeze(1).expand(batch_size, 7)
        mask = labels_exp.eq(classes.expand(batch_size, 7))
        return (distmat * mask.float()).clamp(min=1e-12, max=1e+12).sum() / batch_size

# 4. ResNet50 모델 구조
class FERModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet50(weights='IMAGENET1K_V1')
        in_feats = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()
        self.feature_layer = nn.Sequential(nn.Linear(in_feats, 512), nn.BatchNorm1d(512), nn.ReLU())
        self.classifier = nn.Linear(512, 7)
    def forward(self, x):
        f = self.feature_layer(self.resnet(x))
        return f, self.classifier(f)

# 5. 실행부 (이대로 복붙하십시오)
def train_fer_final_strike():
    logger.info(f"[*] FER 0.5 돌파 작전 개시. (Device: {FERConfig.DEVICE})")
    
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(), transforms.RandomRotation(20), transforms.Resize((224,224)),
        transforms.Grayscale(3), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.5)
    ])
    val_tf = transforms.Compose([
        transforms.Resize((224,224)), transforms.Grayscale(3), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    train_loader = DataLoader(datasets.ImageFolder(FERConfig.TRAIN_DIR, train_tf), 
                              batch_size=FERConfig.BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
    val_loader = DataLoader(datasets.ImageFolder(FERConfig.VAL_DIR, val_tf), 
                            batch_size=FERConfig.BATCH_SIZE, num_workers=8, pin_memory=True)

    model = FERModel().to(FERConfig.DEVICE)
    criterion_ohem = WeightedOHEMLoss().to(FERConfig.DEVICE)
    criterion_center = CenterLoss().to(FERConfig.DEVICE)
    
    opt_m = optim.AdamW(model.parameters(), lr=FERConfig.LR, weight_decay=1e-3)
    opt_c = optim.Adam(criterion_center.parameters(), lr=FERConfig.CENTER_LR)
    
    # 늪 탈출을 위해 학습률을 흔들어주는 스케줄러
    sch = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_m, T_0=10, T_mult=2)

    best_v_loss = float('inf')
    early_stop = 0

    for epoch in range(FERConfig.EPOCHS):
        model.train()
        train_l, correct, total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for imgs, lbls in pbar:
            imgs, lbls = imgs.to(FERConfig.DEVICE), lbls.to(FERConfig.DEVICE)
            opt_m.zero_grad(); opt_c.zero_grad()
            
            feats, outs = model(imgs)
            loss = criterion_ohem(outs, lbls) + (FERConfig.CENTER_WEIGHT * criterion_center(feats, lbls))
            
            loss.backward()
            opt_m.step()
            for p in criterion_center.parameters(): p.grad.data *= (1./FERConfig.CENTER_WEIGHT)
            opt_c.step()
            
            train_l += loss.item()
            _, pred = torch.max(outs, 1); total += lbls.size(0); correct += (pred == lbls).sum().item()
            pbar.set_postfix({'L': f'{loss.item():.4f}', 'Acc': f'{100*correct/total:.1f}%'})

        model.eval(); v_l = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                _, outs = model(imgs.to(FERConfig.DEVICE))
                v_l += F.cross_entropy(outs, lbls.to(FERConfig.DEVICE)).item()
        
        avg_v_l = v_l / len(val_loader)
        logger.info(f"[*] Val Loss: {avg_v_l:.4f}")
        
        sch.step()
        if avg_v_l < best_v_loss:
            best_v_loss = avg_v_l; early_stop = 0
            torch.save(model.state_dict(), FERConfig.SAVE_PATH)
            logger.info("✨ 기록 갱신!")
        else: early_stop += 1
        if early_stop >= FERConfig.PATIENCE: break

if __name__ == "__main__":
    train_fer_final_strike()

2026-03-11 09:57:06,175 [INFO] [*] FER 0.5 돌파 작전 개시. (Device: cuda)
Epoch 1: 100%|█████████████████████████████████████████████████████████████████████████| 449/449 [03:05<00:00,  2.42it/s, L=5.2508, Acc=24.8%]
2026-03-11 10:00:45,355 [INFO] [*] Val Loss: 1.9489
2026-03-11 10:00:45,614 [INFO] ✨ 기록 갱신!
Epoch 2:  48%|███████████████████████████████████▎                                     | 217/449 [01:38<01:27,  2.66it/s, L=3.5303, Acc=29.8%]

# try

In [7]:
import os
import cv2
import torch
import torchvision.transforms as transforms
import numpy as np
from collections import deque

def run_distribution_test(client_id="28", survey_score=80.0):
    print(f"[*] [{client_id}]번 클라이언트 이진 분포도 분석을 시작합니다.")
    
    # 1. 경로 및 환경 설정 (주인님의 환경)
    base_video_dir = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\videos"
    daisee_model_path = "best_focus_model.pth"
    fer_model_path = "best_fer_model.pth"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    emotion_labels = {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'}
    
    daisee_transform = transforms.Compose([
        transforms.ToPILImage(), transforms.Resize((224, 224)),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    fer_transform = transforms.Compose([
        transforms.ToPILImage(), transforms.Resize((224, 224)), transforms.Grayscale(3),
        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 2. 영상 파일 탐색
    client_folder = os.path.join(base_video_dir, str(client_id))
    video_path = None
    if os.path.exists(client_folder):
        for file in os.listdir(client_folder):
            if file.lower().endswith(('.mp4', '.avi', '.mov')):
                video_path = os.path.join(client_folder, file)
                break
                
    if not video_path:
        print(f"[오류] {client_folder} 폴더 내에 영상이 없습니다.")
        return

    # 3. 모델 가중치 로드
    daisee_model = AttentionFocusModel(num_classes=4).to(device)
    fer_model = FERModel(num_classes=7).to(device)
    
    try:
        daisee_model.load_state_dict(torch.load(daisee_model_path, map_location=device))
        fer_model.load_state_dict(torch.load(fer_model_path, map_location=device))
    except Exception as e:
        print(f"[오류] 모델 로드 실패: {e}")
        return

    daisee_model.eval()
    fer_model.eval()

    # 4. 영상 분석 및 분포도 누적
    cap = cv2.VideoCapture(video_path)
    frame_buffer = deque(maxlen=10)
    
    # 분포도를 계산하기 위해 0과 1을 담을 리스트
    fer_binary_distribution = [] 
    all_daisee_scores = []
    
    print("[*] 영상을 화면에 띄웁니다. (종료: 'q')")

    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # --- [FER 이진 분포도 추출] ---
            fer_input = fer_transform(rgb_frame).unsqueeze(0).to(device)
            _, fer_pred = torch.max(fer_model(fer_input).data, 1)
            emotion_idx = fer_pred.item()
            
            # 주인님의 로직: 흥미 있음(1), 흥미 없음(0)
            if emotion_idx in [3, 5, 6]: 
                binary_state = 1
                state_text = "Interested (1)"
                color = (0, 255, 0) # 녹색
            else:
                binary_state = 0
                state_text = "Not Interested (0)"
                color = (0, 0, 255) # 붉은색
                
            fer_binary_distribution.append(binary_state)
            
            # --- [DAiSEE 집중도 추출] ---
            daisee_tensor = daisee_transform(rgb_frame)
            frame_buffer.append(daisee_tensor)
            daisee_level = 0
            
            if len(frame_buffer) == 10:
                seq_input = torch.stack(list(frame_buffer)).unsqueeze(0).to(device)
                _, daisee_pred = torch.max(daisee_model(seq_input).data, 1)
                daisee_level = daisee_pred.item()
                daisee_score = (daisee_level + 1) * 25 if daisee_level > 0 else 25
                all_daisee_scores.append(daisee_score)
                
            # --- 시각적 피드백 ---
            cv2.putText(frame, f"FER State: {state_text}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(frame, f"DAiSEE Level: {daisee_level}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
            cv2.imshow("Distribution Test", frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    
    # 5. 최종 리포트 및 분포도 계산
    total_frames = len(fer_binary_distribution)
    interested_frames = sum(fer_binary_distribution)
    
    # 0 분모 에러 방지 및 비율 산출
    if total_frames > 0:
        interest_ratio = interested_frames / total_frames
        fer_final_score = interest_ratio * 100
    else:
        interest_ratio = 0
        fer_final_score = 0.0

    avg_daisee = float(np.mean(all_daisee_scores)) if all_daisee_scores else 0.0
    
    # 최종 점수 산출
    final_score = (fer_final_score * 0.3) + (avg_daisee * 0.4) + (survey_score * 0.3)

    print("\n" + "="*50)
    print("✅ 영상 분석 완료! [분포도 기반 최종 리포트]")
    print("="*50)
    print(f"- 클라이언트 ID : {client_id}")
    print(f"- 전체 분석 프레임: {total_frames} 장")
    print(f"- 흥미 상태 프레임: {interested_frames} 장")
    print("-" * 50)
    print(f"👉 감정(FER) 분포도 점수  : {fer_final_score:.2f} 점 (전체 구간의 {interest_ratio*100:.1f}%)")
    print(f"👉 집중(DAiSEE) 평균 점수 : {avg_daisee:.2f} 점")
    print(f"👉 자가진단 설문 점수     : {survey_score:.2f} 점")
    print("-" * 50)
    print(f"🏆 최종 흥미도 합산 점수  : {final_score:.2f} / 100")
    print("="*50)

if __name__ == "__main__":
    run_distribution_test(client_id="28", survey_score=80.0)

[*] [28]번 클라이언트 이진 분포도 분석을 시작합니다.


C:\Users\msi\AppData\Local\Temp\ipykernel_21752\1197557208.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  daisee_model.load_state_dict(torch.load(daisee_model_path, ma

[*] 영상을 화면에 띄웁니다. (종료: 'q')

✅ 영상 분석 완료! [분포도 기반 최종 리포트]
- 클라이언트 ID : 28
- 전체 분석 프레임: 1712 장
- 흥미 상태 프레임: 1398 장
--------------------------------------------------
👉 감정(FER) 분포도 점수  : 81.66 점 (전체 구간의 81.7%)
👉 집중(DAiSEE) 평균 점수 : 78.41 점
👉 자가진단 설문 점수     : 80.00 점
--------------------------------------------------
🏆 최종 흥미도 합산 점수  : 79.86 / 100
